# Quantum Key Distribution (QKD) Lab: The BB84 Protocol

**Tutorial:** 

https://quantum.cloud.ibm.com/learning/en/modules/computer-science/quantum-key-distribution

**Goal:** In this lab, you will build a quantum cryptographic key distribution system. You will simulate Alice sending a key to Bob, perform key sifting, and then simulate an eavesdropper (Eve) to see how quantum mechanics inherently detects interception.


### Dependencies
Run the cell below to load the tools you will need. We will use the `AerSimulator` coupled with `FakeBrisbane` (a mock IBM Quantum backend) to simulate realistic hardware noise.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeBrisbane
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import BackendSamplerV2

# Initialize random number generator and our simulated noisy backend
rng = np.random.default_rng(seed=42)
fake_backend = FakeBrisbane()
simulator = AerSimulator.from_backend(fake_backend)
sampler = BackendSamplerV2(backend=simulator)

bit_num = 25 # Length of our initial message

## Task 1: Alice's Preparation

In the BB84 protocol, Alice wants to send a secret key to Bob. She starts by generating random bits (0s and 1s) and choosing random bases to encode them. 
* We will define `0` as the Z-basis (Standard) and `1` as the X-basis (Hadamard).
* **Instructions:** Write code to randomly generate an array of `bit_num` bits for Alice, and an array of `bit_num` bases.

In [ ]:
# TODO: Generate an array of size `bit_num` containing random 0s and 1s for Alice's bits
alice_bits = # YOUR CODE HERE

# TODO: Generate an array of size `bit_num` containing random 0s and 1s for Alice's bases
alice_bases = # YOUR CODE HERE

print("Alice's Bits: ", alice_bits)
print("Alice's Bases:", alice_bases)

## Task 2: Encoding the Quantum Circuit
Now, Alice encodes her bits into qubits. 
* If Alice's bit is `1`, she applies an `X` gate to flip the qubit to $|1\rangle$.
* If Alice's basis is `1` (X-basis), she applies an `H` gate to move the qubit into the $|+\rangle$ or $|-\rangle$ state.

**Instructions:** Iterate through the `bit_num` qubits and apply the correct gates based on `alice_bits` and `alice_bases`.

In [ ]:
qc = QuantumCircuit(bit_num, bit_num)

for i in range(bit_num):
    # TODO: Apply an X gate if alice_bits[i] is 1
    # YOUR CODE HERE
    
    # TODO: Apply an H gate if alice_bases[i] is 1
    # YOUR CODE HERE

qc.barrier()
qc.draw()

## Task 3: Bob's Measurement
Bob receives the qubits, but he doesn't know Alice's bases. He must guess!
**Instructions:** Generate an array of random bases for Bob. Then, if Bob's base is `1`, apply an `H` gate before measuring to rotate the measurement into the X-basis.

In [ ]:
# TODO: Generate Bob's random bases
bob_bases = # YOUR CODE HERE

for i in range(bit_num):
    # TODO: If bob_bases[i] is 1, apply an H gate to qubit i
    # YOUR CODE HERE
    
    # TODO: Measure qubit i into classical bit i
    # YOUR CODE HERE

qc.draw('mpl')

## Task 4: Execution & Key Sifting
We will now transpile the circuit for our `FakeBrisbane` simulator and run it. Afterward, Alice and Bob compare their bases over a public channel. If their bases match, they keep the bit. If they differ, they discard it.

In [ ]:
# Transpile and run the circuit
pm = generate_preset_pass_manager(target=simulator.target, optimization_level=3)
qc_transpiled = pm.run(qc)
job = sampler.run([qc_transpiled], shots=1)

# Extract results
counts = job.result()[0].data.c.get_counts()
bitstring = list(counts.keys())[0]

# Convert bitstring to list of integers (reverse it because Qiskit uses little-endian ordering)
bob_measured_bits = [int(b) for b in bitstring][::-1]

# TODO: Key Sifting
alice_sifted_key = []
bob_sifted_key = []

for i in range(bit_num):
    # TODO: Check if Alice and Bob's bases match. If they do, append the respective bits to their sifted keys.
    # YOUR CODE HERE

# Calculate Fidelity
matches = sum(1 for a, b in zip(alice_sifted_key, bob_sifted_key) if a == b)
fidelity = matches / len(alice_sifted_key) if len(alice_sifted_key) > 0 else 0

print(f"Alice's Key: {alice_sifted_key}")
print(f"Bob's Key:   {bob_sifted_key}")
print(f"Fidelity:    {fidelity * 100:.2f}%")

## Task 5: Enter Eve the Eavesdropper
What happens if someone intercepts the transmission? Because of the **No-Cloning Theorem**, Eve cannot copy the qubit. She must measure it and send a new qubit to Bob based on her measurement.

**Instructions:**
1. Create a new `QuantumCircuit`.
2. Encode Alice's states as you did in Task 2.
3. Insert Eve's interception: Generate random bases for Eve, apply H gates if her basis is 1, measure the qubit, and prepare a new qubit to send to Bob.
4. Apply Bob's measurements as you did in Task 3.
5. Run the circuit and calculate the new fidelity. What happened to the error rate?

In [ ]:
# TODO: Implement the Intercept-Resend Attack!
# Build the circuit, run it, sift the keys, and calculate fidelity.
# You should see the fidelity drop < 100%.

## Task 6: Using the Key as a One-Time Pad
Now that Alice and Bob share a secure key, Alice wants to send Bob a secret binary message. 

She will encrypt her message using an **XOR** operation (`^` in Python). She will send the encrypted message over a public channel, and Bob will use his identical key to decrypt it with the exact same XOR operation.

*Note: The message cannot be longer than the sifted key.*